In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = "/content/drive/MyDrive/BTU_RM_SoSe26/data/harmonixset"
DRIVE_OUT  = "/content/drive/MyDrive/BTU_RM_SoSe26/outputs"
os.makedirs(DRIVE_DATA, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

In [ ]:
import os, sys, json, random, time, csv, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from notebooks.utils import *

!pip install -q . 2>&1 | tail -5

os.makedirs("data", exist_ok=True)
if not os.path.isdir("data/harmonixset"):
    !git clone https://github.com/urinieto/harmonixset.git data/harmonixset

for sub in ["audio", "features"]:
    src = os.path.join(DRIVE_DATA, sub)
    dst = os.path.join("data/harmonixset", sub)
    os.makedirs(src, exist_ok=True)
    if not os.path.islink(dst):
        !ln -s "{src}" "{dst}"
if not os.path.islink("outputs"):
    !ln -s "{DRIVE_OUT}" outputs

print(f"Annotations: {len(os.listdir('data/harmonixset/dataset/segments'))} segment files")

In [ ]:
songs = get_songs_with_audio()
print(f"Songs with audio: {len(songs)}")
precompute_features(songs, force=False)
print(f"Features ready: {sum(1 for s in songs if os.path.exists(f'data/harmonixset/features/{s}.npy'))}/{len(songs)}")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

TARGET_HOP_TIME = 6 * 512 / 16000
CHUNK_FRAMES = int(24.0 / TARGET_HOP_TIME)
CHUNK_HOP_FRAMES = int(3.0 / TARGET_HOP_TIME)
BATCH_SIZE = 128; MAX_EPOCHS = 100; BATCHES_PER_EPOCH = 500
MAX_GRAD_NORM = 1.0; LR = 0.0005; WEIGHT_DECAY = 0.9

songs = get_songs_with_audio()
precompute_features(songs, force=False)
annotations = load_all_annotations()
print(f"Songs: {len(songs)}, Annotations: {len(annotations)}")

print("Loading features into RAM...")
features = {sid: load_features(sid) for sid in songs}
print("Done")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = 10000 ** (torch.arange(0, d_model, 2).float() / d_model)
        pe[:, 0::2] = torch.sin(pos / div); pe[:, 1::2] = torch.cos(pos / div)
        self.register_buffer("pe", pe)
    def forward(self, x):
        return x + self.pe[:x.size(-2), :]

class SpectralEncoder(nn.Module):
    def __init__(self, d_model=96, nhead=4, dim_feedforward=384, dropout=0.1):
        super().__init__()
        self.norm_in = nn.LayerNorm(d_model)
        self.pos_enc = PositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, activation="relu", batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=1)
    def forward(self, x):
        B, C, F, T = x.shape
        x = x.permute(0, 3, 2, 1).reshape(B * T, F, C)
        x = self.norm_in(x); x = self.pos_enc(x); x = self.encoder(x)
        return x.reshape(B, T, F, C).permute(0, 3, 2, 1)

class TemporalEncoder(nn.Module):
    def __init__(self, d_model=96, nhead=8, dim_feedforward=384, dropout=0.1):
        super().__init__()
        self.norm_in = nn.LayerNorm(d_model)
        self.pos_enc = PositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, activation="relu", batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=1)
    def forward(self, x):
        B, C, F, T = x.shape
        x = x.permute(0, 2, 3, 1).reshape(B * F, T, C)
        x = self.norm_in(x); x = self.pos_enc(x); x = self.encoder(x)
        return x.reshape(B, F, T, C).permute(0, 3, 1, 2)

class SpecTNTBlock(nn.Module):
    def __init__(self, d_model=96):
        super().__init__()
        self.spectral = SpectralEncoder(d_model)
        self.temporal = TemporalEncoder(d_model)
        self.norm_spec = nn.LayerNorm(d_model)
        self.norm_temp = nn.LayerNorm(d_model)
    def forward(self, x):
        B, C, F, T = x.shape
        spec = self.spectral(x)
        x = self.norm_spec((x + spec).permute(0, 2, 3, 1)).permute(0, 3, 1, 2)
        temp = self.temporal(x)
        x = self.norm_temp((x + temp).permute(0, 2, 3, 1)).permute(0, 3, 1, 2)
        return x

class ResNetFrontend(nn.Module):
    def __init__(self, in_ch=1, out_ch=96):
        super().__init__()
        self.net = nn.Sequential(
            nn.Sequential(nn.Conv2d(in_ch, 32, 3, (1, 1), 1), nn.BatchNorm2d(32), nn.ReLU()),
            nn.Sequential(nn.Conv2d(32, 64, 3, (2, 1), 1), nn.BatchNorm2d(64), nn.ReLU()),
            nn.Sequential(nn.Conv2d(64, 64, 3, (1, 1), 1), nn.BatchNorm2d(64), nn.ReLU()),
            nn.Sequential(nn.Conv2d(64, 128, 3, (2, 1), 1), nn.BatchNorm2d(128), nn.ReLU()),
            nn.Sequential(nn.Conv2d(128, 128, 3, (1, 1), 1), nn.BatchNorm2d(128), nn.ReLU()),
            nn.Sequential(nn.Conv2d(128, 128, 3, (1, 1), 1), nn.BatchNorm2d(128), nn.ReLU()),
            nn.Sequential(nn.Conv2d(128, out_ch, 3, (1, 1), 1), nn.BatchNorm2d(out_ch), nn.ReLU()),
        )
    def forward(self, x):
        return self.net(x.unsqueeze(1))

class SpecTNT(nn.Module):
    def __init__(self, d_model=96, n_blocks=5):
        super().__init__()
        self.frontend = ResNetFrontend(out_ch=d_model)
        self.blocks = nn.ModuleList([SpecTNTBlock(d_model) for _ in range(n_blocks)])
        self.output = nn.Linear(d_model, 8)
    def forward(self, x):
        x = self.frontend(x)
        for block in self.blocks:
            x = block(x)
        x = x.mean(dim=2).permute(0, 2, 1)
        return self.output(x)

def pad_tokens(tokens_list, device):
    max_len = max((len(t) for t in tokens_list), default=1)
    if max_len == 0: max_len = 1
    padded = torch.zeros((len(tokens_list), max_len), dtype=torch.long, device=device)
    lengths = torch.zeros(len(tokens_list), dtype=torch.long, device=device)
    for i, t in enumerate(tokens_list):
        tl = torch.tensor(t, device=device)
        padded[i, :len(tl)] = tl
        lengths[i] = len(tl)
    return padded, lengths

os.makedirs("outputs/models", exist_ok=True)
os.makedirs("outputs/results", exist_ok=True)
n = len(songs); fold_size = n // 4
song_ids = np.array(songs)
all_results = {}

for fold in range(4):
    val_set = set(song_ids[fold * fold_size:(fold + 1) * fold_size])
    train_ids = [s for s in songs if s not in val_set]
    val_ids = [s for s in songs if s in val_set]
    print(f"\n{'='*60}\nFold {fold}: {len(train_ids)} train, {len(val_ids)} val\n{'='*60}")

    chunks = [(sid, s) for sid in train_ids for s in range(0, features[sid].shape[1] - CHUNK_FRAMES + 1, CHUNK_HOP_FRAMES) if features[sid].shape[1] >= CHUNK_FRAMES]

    model = SpecTNT().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=2, factor=0.5)
    best_loss = float("inf"); patience = 0

    for epoch in range(MAX_EPOCHS):
        model.train(); random.shuffle(chunks)
        total = min(len(chunks), BATCHES_PER_EPOCH * BATCH_SIZE)
        pbar = tqdm(range(0, total, BATCH_SIZE), desc=f"Epoch {epoch}", leave=False)
        epoch_loss = 0
        for i in pbar:
            batch = chunks[i:i+BATCH_SIZE]
            fts = torch.stack([torch.from_numpy(features[s][:, s2:s2+CHUNK_FRAMES]).float() for s, s2 in batch]).to(device)
            b = torch.stack([torch.from_numpy(annotations[s]["boundary"][s2:s2+CHUNK_FRAMES]).float() for s, s2 in batch]).to(device)
            fn = torch.stack([torch.from_numpy(annotations[s]["functions"][s2:s2+CHUNK_FRAMES]).float() for s, s2 in batch]).to(device)
            tokens = [get_section_tokens(annotations[s]["segments"]) for s, _ in batch]
            opt.zero_grad()
            out = model(fts)
            loss = 0.9 * F.binary_cross_entropy_with_logits(out[:, :, 0], b) + 0.1 * F.binary_cross_entropy_with_logits(out[:, :, 1:], fn)
            if any(len(t) > 0 for t in tokens):
                padded, lengths = pad_tokens(tokens, device)
                ctl = ctl_loss_batch(out[:, :, 1:], padded, lengths)
                loss = loss + 0.1 * ctl
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM); opt.step()
            epoch_loss += loss.item() * len(batch)

        model.eval()
        v_chunks = [(sid, s) for sid in val_ids for s in range(0, features[sid].shape[1] - CHUNK_FRAMES + 1, CHUNK_HOP_FRAMES) if features[sid].shape[1] >= CHUNK_FRAMES]
        v_loss = 0
        with torch.no_grad():
            for i in range(0, len(v_chunks), BATCH_SIZE):
                batch = v_chunks[i:i+BATCH_SIZE]
                fts = torch.stack([torch.from_numpy(features[s][:, s2:s2+CHUNK_FRAMES]).float() for s, s2 in batch]).to(device)
                b = torch.stack([torch.from_numpy(annotations[s]["boundary"][s2:s2+CHUNK_FRAMES]).float() for s, s2 in batch]).to(device)
                fn = torch.stack([torch.from_numpy(annotations[s]["functions"][s2:s2+CHUNK_FRAMES]).float() for s, s2 in batch]).to(device)
                out = model(fts)
                loss = 0.9 * F.binary_cross_entropy_with_logits(out[:, :, 0], b) + 0.1 * F.binary_cross_entropy_with_logits(out[:, :, 1:], fn)
                v_loss += loss.item() * len(batch)
        v_loss /= max(len(v_chunks), 1); epoch_loss /= max(total, 1)
        sched.step(v_loss)
        print(f"Epoch {epoch:3d} | train: {epoch_loss:.4f} | val: {v_loss:.4f} | lr: {opt.param_groups[0]['lr']:.2e}")

        if v_loss < best_loss:
            best_loss = v_loss; patience = 0
            torch.save(model.state_dict(), f"outputs/models/SpecTNT_CTL_fold{fold}.pt")
        else:
            patience += 1
            if patience >= 4: print(f"Early stopping at epoch {epoch}"); break

    model.load_state_dict(torch.load(f"outputs/models/SpecTNT_CTL_fold{fold}.pt", weights_only=True))
    model.eval()
    fold_metrics = []
    for sid in val_ids:
        feat = features[sid]; T = feat.shape[1]
        bc = np.zeros(T); fc = np.zeros((T, 7)); cnt = np.zeros(T)
        pred_hop = CHUNK_HOP_FRAMES
        with torch.no_grad():
            for start in range(0, T - CHUNK_HOP_FRAMES + 1, pred_hop):
                end = min(start + CHUNK_FRAMES, T)
                ft = torch.from_numpy(feat[:, start:end]).float().unsqueeze(0).to(device)
                out = model(ft)
                n = min(out.shape[1], end - start)
                bc[start:start+n] += out[0, :n, 0].cpu().numpy()
                fc[start:start+n] += out[0, :n, 1:].cpu().numpy()
                cnt[start:start+n] += 1
        cnt = np.maximum(cnt, 1)
        segs = postprocess_song(bc / cnt, fc / cnt[:, np.newaxis], annotations[sid]["duration"])
        fold_metrics.append(compute_metrics(annotations[sid]["segments"], segs, annotations[sid]["duration"]))

    avg = {k: float(np.mean([m[k] for m in fold_metrics])) for k in fold_metrics[0]}
    all_results[f"fold_{fold}"] = avg
    print(f"\nFold {fold}:")
    for k, v in avg.items(): print(f"  {k}: {v:.3f}")

overall = {k: float(np.mean([all_results[f][k] for f in all_results])) for k in list(all_results.values())[0]}
print(f"\n{'='*60}\nSpecTNT+CTL Average:")
for k, v in overall.items(): print(f"  {k}: {v:.3f}")
json.dump({"per_fold": all_results, "average": overall}, open("outputs/results/SpecTNT_CTL.json", "w"), indent=2)
print(f"\nResults saved to outputs/results/SpecTNT_CTL.json")

In [ ]:
import glob, json, pandas as pd
results = sorted(glob.glob("outputs/results/*.json"))
for path in results:
    name = os.path.basename(path).replace(".json", "")
    data = json.load(open(path))
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    df = pd.DataFrame(data["per_fold"]).T
    df.index = [f"Fold {i}" for i in range(len(df))]
    print(df.to_string())
    print(f"\nAverage: {data['average']}")